[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashakram05/ayeshaAkram-flyrank/blob/main/work/notebooks/w03_data_contract.ipynb)

# ML-04 — Search Intelligence Data Contract

## Lane 2: Refresh / Content Opportunity Scoring

This notebook defines the data contract for the content opportunity scoring problem.

The goal is to identify which content items deserve limited human review first. The data contract therefore needs to make clear:

- what one row represents
- which time window is being used
- which fields can be used as predictive features
- which fields are only context
- which fields must be excluded because they create leakage or are not meaningful predictors
- what limitations the data has

The contract is designed to support a later ranking/scoring model while keeping the analysis honest.

## 1. Unit of analysis + time window

### Unit of Analysis

One row represents the daily performance of one pseudonymized content item for one pseudonymized client on a single report date.

The intended modeling unit is therefore:

> content item × client × report date

### Time Window

For this notebook, I use the March 2026 partition:

`month = '2026-03'`

A mid-panel month is used for verification rather than treating the final available period as the natural outcome window.

The large warehouse is queried remotely through DuckDB rather than loading the full dataset into pandas.

In [1]:
!pip -q install duckdb huggingface_hub

In [2]:
from google.colab import userdata
import duckdb

# Connect to DuckDB
con = duckdb.connect()

# Hugging Face token should be stored in Colab Secrets.
# Secret name used in this notebook: flyrank
hf_token = userdata.get("flyrank")

if not hf_token:
    raise ValueError(
        "Hugging Face token not found. Add your token to Colab Secrets "
        "with the name 'flyrank'."
    )

# Register Hugging Face credentials with DuckDB.
con.execute(f"""
CREATE OR REPLACE SECRET flyrank_hf (
    TYPE huggingface,
    TOKEN '{hf_token}'
)
""")

# Dataset location
rel = "hf://datasets/FlyRank/internship-warehouse"

# March 2026 partition
march_path = (
    f"{rel}/fact_content_daily_performance/month=2026-03/*.parquet"
)

print("DuckDB connection ready.")
print("Analysis partition:", march_path)

DuckDB connection ready.
Analysis partition: hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet


### Listing Available Partitions in Hugging Face Dataset

To find the correct `month=` partitions, we can use the `huggingface_hub` library to inspect the dataset repository directly. First, we need to install it.

## 2. Fields: feature / label / context / excluded


In [3]:
# Verify the analysis window and row count.

summary = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_report_date,
    MAX(report_date) AS last_report_date
FROM read_parquet('{march_path}')
""").df()

display(summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,first_report_date,last_report_date
0,9841378,2026-03-01,2026-03-31


### Verification result

The March 2026 partition contains **9,841,378 rows** covering **2026-03-01 through 2026-03-31**.

Because the dataset is large, I query it remotely with DuckDB and only bring small aggregated results into pandas. This avoids loading millions of raw rows into notebook memory.

## 2. Fields: feature / label / context / excluded

### Candidate Features

The following observed performance variables are candidates for predictive features:

- `gsc_impressions`
- `gsc_clicks`
- `gsc_avg_position`
- `ga4_sessions`
- `ga4_users`

These describe observed search and analytics performance available at or before the report date.

Other content/freshness variables can be considered if they are present in the relevant modeling dataset and satisfy the same decision-time availability requirement.

### Label / Proxy

The ideal outcome for the content opportunity problem would be a future observed outcome showing whether a page benefited from an intervention or demonstrated a useful future performance change.

That direct outcome is not defined in this data contract.

For development, Week 2 identified `trend_direction = "down"` in the starter dataset as a temporary proxy for observed decline. That proxy is not treated as ground truth for whether a page truly needs a refresh.

The final modeling outcome must be defined separately from the feature contract.

### Context

The following fields are used for grouping, filtering, validation, or interpretation:

- `client_hash_id`
- `content_hash_id`
- `report_date`
- `month`

These identifiers should not be used directly as predictive features.

### Excluded

The following information is excluded from predictive features:

- future performance metrics, because they would introduce target leakage
- hash identifiers, because they identify entities rather than represent meaningful content signals
- GA4 values from periods where `ga4_data_available` is false, when creating GA4-based features

The GA4 availability flag must be respected because zero-filled values before GA4 collection do not necessarily mean zero user activity.

In [5]:
fields = {
    "Feature": [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_sessions",
        "ga4_users"
    ],
    "Label / Proxy": [
        "Future observed performance outcome; development proxy may be used separately"
    ],
    "Context": [
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "month"
    ],
    "Excluded": [
        "Future performance metrics",
        "Hash identifiers as predictive features",
        "GA4 values where ga4_data_available is false"
    ]
}

import pandas as pd

fields_df = pd.DataFrame({
    "Category": [
        "Features",
        "Features",
        "Features",
        "Features",
        "Features",
        "Label / Proxy",
        "Context",
        "Context",
        "Context",
        "Context",
        "Excluded",
        "Excluded",
        "Excluded"
    ],
    "Field": [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_sessions",
        "ga4_users",
        "Future observed performance / development proxy",
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "month",
        "Future performance metrics",
        "Hash identifiers",
        "GA4 values when ga4_data_available is false"
    ]
})

display(fields_df)

,Category,Field
0,Features,gsc_impressions
1,Features,gsc_clicks
2,Features,gsc_avg_position
3,Features,ga4_sessions
4,Features,ga4_users
5,Label / Proxy,Future observed performance / development proxy
6,Context,client_hash_id
7,Context,content_hash_id
8,Context,report_date
9,Context,month


## 3. Verify it with queries

Every important contract claim should be supported by a query.

The checks below verify:

1. the documented grain
2. row count and date window
3. GSC and GA4 availability
4. whether the candidate fields actually exist
5. missing-value behavior

In [6]:
# Query 1 — Verify the grain

grain = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS duplicate_rows
FROM read_parquet('{march_path}')
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""").df()

print("Query 1: Grain Verification")
display(grain)

if grain.empty:
    print(
        "✓ No duplicate report_date-client-content combinations "
        "were found in the checked March partition."
    )
else:
    print(
        "⚠ Duplicate combinations were found. "
        "The documented grain needs further investigation."
    )

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 1: Grain Verification


,report_date,client_hash_id,content_hash_id,duplicate_rows


✓ No duplicate report_date-client-content combinations were found in the checked March partition.


In [7]:
# Query 2 — Verify row count and date window

counts = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet('{march_path}')
""").df()

print("Query 2: Row Count and Date Window")
display(counts)

Query 2: Row Count and Date Window


,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [8]:
# Query 3 — Verify GSC and GA4 data availability

availability = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows,
    COUNT(*) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS ga4_available_rows
FROM read_parquet('{march_path}')
""").df()

print("Query 3: Data Availability")
display(availability)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 3: Data Availability


,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061,413966


### Interpretation of availability

The March partition does not have GSC and GA4 coverage for every row.

Therefore, the absence of a GA4 value must not automatically be interpreted as zero engagement.

For later modeling:

- GSC-based features should respect `gsc_data_available`.
- GA4-based features should respect `ga4_data_available`.
- Availability flags can be retained as context or explicit features where appropriate.
- Zero-filled values from periods before a data source became available should not be treated as real behavioral measurements.

In [9]:
# Query 4 — Inspect the schema for candidate fields

schema = con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet('{march_path}')
""").df()

display(schema[["column_name", "column_type"]])

,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


In [10]:
# Check whether the expected candidate fields are present.

candidate_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_users",
    "gsc_ctr"
]

available = [
    feature
    for feature in candidate_features
    if feature in schema["column_name"].tolist()
]

missing = [
    feature
    for feature in candidate_features
    if feature not in schema["column_name"].tolist()
]

print("Available candidate fields:")
print(available)

print("\nFields not found in the March schema:")
print(missing)

Available candidate fields:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ga4_users']

Fields not found in the March schema:
['gsc_ctr']


## 7. Data limitations

This warehouse records observed search and analytics performance, but it does not explain why rankings, traffic, or engagement changed.

Important limitations include:

### 1. Observational data

The data shows what happened, but it does not establish that a content change caused a later performance change.

### 2. Unbalanced histories

Clients begin appearing in the warehouse at different times, so their histories are not equally long.

### 3. Uneven source availability

GSC and GA4 are not available for every row. Availability flags must therefore be respected.

### 4. Proxy outcome limitation

The development-time decline proxy is not the same as a true label for whether a page needs a refresh.

### 5. Human decision remains necessary

A high opportunity score is a prioritization signal, not proof that a particular editorial action will improve performance.

These limitations mean the model should be presented as **decision support**, not as causal evidence or an automatic content decision system.

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.